In [10]:
import numpy as np
import pandas as pd
import cupy as cp
import cupyx
import tqdm
import torch
from typing import Dict
from utils import utils
from utils.data_utils import load_metadata_and_map

In [11]:
METADATA_FILE = "embeddings/id_metadata.json"
TENSOR_DIR = "embeddings/uuid_embeddings/embeddings/"

In [12]:
# 1. Load Tensors
tensors = utils.load_tensor(TENSOR_DIR, num_workers=16)


Loading cache from embeddings/cache


In [13]:
# 2. Load Metadata (Shared Logic)
metadata_df, artist_to_track_idx = load_metadata_and_map(METADATA_FILE, tensors)

Loading metadata from embeddings/id_metadata.json...
Building Artist to Track Index map...


In [23]:
# 3. Pool Embeddings
print("Pooling embeddings (mean+max)...")
all_embeddings = utils.pool_loaded_tensor_dict(tensors=tensors, mode='mean+max')

embeddings_torch = torch.stack(list(all_embeddings.values())).cuda()

all_embeddings_gpu = cp.asarray(embeddings_torch)


Pooling embeddings (mean+max)...
Detected allchunks tensors, pooling to 1D embedding (mean+max)


100%|██████████| 138748/138748 [00:03<00:00, 41165.00it/s]


In [24]:
# 4. Load VLAD and Centroid results
vlad_results = pd.read_json("data/validation/vlad_results.jsonl", orient="records", lines=True)
centroid_results = pd.read_json("data/validation/centroid_results.jsonl", orient="records", lines=True)


In [ ]:
# 5. Evaluate based on pairwise artist distance to average artist distance
def compute_pairwise_artist_distance(artist_a_idx, artist_b_idx, embeddings):
  """
  Computes the average distance between all track pairs of two artists.
  """
  # Get all track vectors for Artist A and Artist B
  tracks_a = embeddings[artist_a_idx]
  tracks_b = embeddings[artist_b_idx]
  
  # Compute distance matrix between A's tracks and B's tracks
  # shape: (num_tracks_a, num_tracks_b)
  # Uses ||a - b||^2 = ||a||^2 + ||b||^2 - 2<a, b>
  
  tracks_a = cp.array(tracks_a)
  tracks_b = cp.array(tracks_b)

  norm_a = cp.sum(tracks_a**2, axis=1, keepdims=True)
  norm_b = cp.sum(tracks_b**2, axis=1) # broadcastable
  dot = cp.dot(tracks_a, tracks_b.T)
  
  dists_sq = norm_a + norm_b - 2 * dot
  dists = cp.sqrt(cp.maximum(dists_sq, 0))
  
  # The true audio distance is the mean of all pair distances
  return cp.mean(dists).item()

vlad_distances: Dict[str, float] = {}
for _, row in tqdm.tqdm(vlad_results.iterrows()):
  artist_a_idx = row["ArtistID"]
  matched_artists = row["Results"]
  dist = []

  track_a_idx = artist_to_track_idx[artist_a_idx]

  for artist_b_idx in matched_artists:
    track_idx_b = artist_to_track_idx[artist_b_idx]
    distance = compute_pairwise_artist_distance(cp.asarray(track_a_idx), cp.asarray(track_idx_b), all_embeddings_gpu)
    dist.append(distance)
    # print(distance)

  # Compute the average distance to all matched artists
  vlad_distances[artist_a_idx] = np.mean(dist)

centroid_distances: Dict[str, float] = {}
for _, row in tqdm.tqdm(centroid_results.iterrows()):
  artist_a_idx = row["ArtistID"]
  matched_artists = row["Results"]
  dist = []

  track_a_idx = artist_to_track_idx[artist_a_idx]

  for artist_b_idx in matched_artists:
    track_idx_b = artist_to_track_idx[artist_b_idx]
    distance = compute_pairwise_artist_distance(cp.asarray(track_a_idx), cp.asarray(track_idx_b), all_embeddings_gpu)
    dist.append(distance)
    # print(distance)

  # Compute the average distance to all matched artists
  centroid_distances[artist_a_idx] = np.mean(dist)


994it [00:03, 252.21it/s]
994it [00:03, 258.33it/s]


In [ ]:
# 6. Compute the average distance to all artists
vlad_avg_distance = np.mean(list(vlad_distances.values()))
centroid_avg_distance = np.mean(list(centroid_distances.values()))

print(f"VLAD Average Distance: {vlad_avg_distance}")
print(f"Centroid Average Distance: {centroid_avg_distance}")


VLAD Average Distance: 0.28850534607459427
Centroid Average Distance: 0.28566550054023265
